# `main.py` — Point d'entrée du projet

> **MODULE FOURNI — NE PAS MODIFIER**

Ce notebook enchaîne tout le pipeline :

1. **Chargement** des marques connues et des courriels de test ;
2. **Analyse** de chaque courriel par `scoring_engine` ;
3. **Évaluation** des verdicts avec `evaluation` ;
4. **Et maintenant ?** — les prochaines étapes.

| Script d'origine | Équivalent dans ce notebook |
|---|---|
| `python3 main.py` | `DETAIL = False` puis exécuter `main()` |
| `python3 main.py --detail` | `DETAIL = True`, `LIMITE = 3` |
| `python3 main.py --detail 5` | `DETAIL = True`, `LIMITE = 5` |

## Imports

On s'appuie sur les modules du projet :
- `phishing_rules` (module 1) fournit `charger_domaines_officiels()`
- `scoring_engine` fournit `analyser_courriel()` et `afficher_rapport()`
- `evaluation` fournit `rapport()` et `montrer_erreurs()`

In [ ]:
import csv

import evaluation
import phishing_rules
import scoring_engine

---
## Chemins des données

| Constante | Valeur | Contenu |
|---|---|---|
| `CHEMIN_DONNEES` | `donnees/emails_test.csv` | le jeu de courriels étiquetés |
| `CHEMIN_MARQUES` | `domaines_officiels.csv` | les domaines officiels des marques |

In [ ]:
CHEMIN_DONNEES = "donnees/emails_test.csv"
CHEMIN_MARQUES = "domaines_officiels.csv"

---
## Options d'affichage

Dans le script, ces options venaient de la ligne de commande (`sys.argv`).
Dans un notebook, on les règle directement ici :

- `DETAIL` : afficher en plus le rapport de chaque courriel ;
- `LIMITE` : nombre de rapports détaillés affichés (3 par défaut).

In [ ]:
DETAIL = False   # True <=> python3 main.py --detail
LIMITE = 3       # nombre de courriels détaillés si DETAIL = True

---
## `charger_courriels(chemin)`

Lit le CSV et renvoie une **liste de dictionnaires** : une entrée par courriel,
une clé par colonne (`sender`, `subject`, `label`, …).

In [ ]:
def charger_courriels(chemin=CHEMIN_DONNEES):
    """Lit le CSV et renvoie une liste de dictionnaires."""
    with open(chemin, encoding="utf-8") as f:
        return list(csv.DictReader(f))

---
## `main(detail, limite)`

Le pipeline complet, en quatre étapes :

| Étape | Rôle |
|---|---|
| **[1] Chargement** | lit les marques et les courriels, affiche la répartition des étiquettes |
| **[2] Analyse** | calcule un verdict pour chaque courriel |
| **[3] Évaluation** | matrice de confusion, mesures par classe, coût, erreurs |
| **[4] Et maintenant ?** | rappel des prochaines étapes |

Si une fonction n'est pas encore écrite, elle renvoie `None` et l'analyse lève
une `TypeError` : le message invite alors à lancer `verification`.

In [ ]:
def main(detail=DETAIL, limite=LIMITE):
    print("\n[1] Chargement")
    table_marques = phishing_rules.charger_domaines_officiels(CHEMIN_MARQUES)
    courriels = charger_courriels()
    print(f"    {len(table_marques)} marques connues")
    print(f"    {len(courriels)} courriels charges")

    repartition = {}
    for c in courriels:
        repartition[c["label"]] = repartition.get(c["label"], 0) + 1
    print("    repartition :", repartition)

    print("\n[2] Analyse")
    try:
        resultats = [scoring_engine.analyser_courriel(c, table_marques)
                     for c in courriels]
    except TypeError as err:
        print("\n  /!\\  Une fonction renvoie None : elle n'est pas encore ecrite.")
        print(f"       Detail Python : {err}")
        print("       Lancez le notebook verification pour voir lesquelles.")
        return

    verdicts = [r["verdict"] for r in resultats]
    reels = [c["label"] for c in courriels]

    if detail:
        print()
        for r in resultats[:limite]:
            scoring_engine.afficher_rapport(r)
            print()

    print("\n[3] Evaluation")
    evaluation.rapport(reels, verdicts, "Regles Python (module 1 + module 2)")
    evaluation.montrer_erreurs(courriels, reels, verdicts)

    print("\n[4] Et maintenant ?")
    print("    - Ajustez les seuils dans scoring_engine, relancez,")
    print("      et notez le avant/apres. C'est un livrable note.")
    print("    - Passez ensuite a la phase IA (voir le guide etudiant).")

---
## Lancement

Exécutez la cellule ci-dessous. Pour voir le détail de chaque courriel,
passez `DETAIL = True` plus haut (ou appelez `main(detail=True, limite=5)`).

In [ ]:
main()